# Notebook 01: Exploratory Data Analysis & DHS Indicator Inspection
**Project**: Maternal-Child Health Profiles Across Countries Through Unsupervised Clustering  
**Source**: Demographic and Health Surveys (DHS) Program API

### Objectives:
1. Load raw multi-country indicator records and survey metadata.
2. Filter for standard 5-year retrospective national estimates.
3. Construct the latest cross-sectional dataset per country ($SurveyYear \ge 2000$).
4. Inspect dimensions, data types, and preliminary missingness patterns.

In [ ]:
import sys
import os
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.data_loader import load_raw_data, select_standard_records, get_latest_cross_sectional_dataset, get_dataset_summary

### 1. Ingest Raw DHS Indicators & Metadata

In [ ]:
raw_df, meta_df = load_raw_data(config.RAW_DATA_FILE, config.SURVEY_META_FILE)
print(f"Raw indicator records: {len(raw_df):,}")
print(f"Available surveys: {len(meta_df):,}")
raw_df.head(5)

### 2. Standardize Reference Windows & Construct Cross-Section

In [ ]:
cross_df = get_latest_cross_sectional_dataset(min_year=2000)
summary = get_dataset_summary(cross_df)
print(f"Constructed cross-sectional matrix: {summary['num_countries']} countries x {summary['indicators_count']} indicators")
print(f"Overall missingness: {summary['missing_percentage']:.2f}%")
cross_df.head()

### 3. Missingness Matrix Across Countries

In [ ]:
feat_cols = [c for c in config.FEATURE_COLUMNS if c in cross_df.columns]
missing_counts = cross_df[feat_cols].isna().sum()
missing_pct = (missing_counts / len(cross_df)) * 100

missing_summary = pd.DataFrame({"Missing Count": missing_counts, "Missing %": missing_pct})
print(missing_summary)

plt.figure(figsize=(12, 6))
sns.heatmap(cross_df.set_index("CountryName")[feat_cols].isna().T, cmap="Blues", cbar=True)
plt.title("Missingness Pattern Matrix across Countries and Health Indicators")
plt.xlabel("Countries")
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()